**AI Powered Resume Screening & Candidate Selection Using Machine Learning**

**Introduction**

Resume screening and candidate selection refer to the process of evaluating job applicants based on their qualifications, skills, and experience to determine their suitability for a role.

In today’s recruitment process, companies receive a large number of resumes, making manual screening time-consuming and inefficient.

Using artificial intelligence and machine learning, this process can be automated to assist recruiters in making faster, consistent, and data-driven decisions.

AI-powered screening helps companies:

• Identify the most suitable candidates efficiently

• Reduce manual effort in resume screening

• Improve hiring accuracy and consistency

• Enhance overall recruitment productivity

In this project, we develop a machine learning model that predicts whether a candidate should be selected or rejected based on features such as skills, experience, projects, education, job role, and salary expectation extracted from resumes.


**Business Problem**

In the recruitment process, manually screening a large number of resumes is time-consuming, inconsistent, and prone to human bias. Companies often struggle to quickly identify the most suitable candidates from a large pool of applicants.

The goal of this project is to build a predictive model that can classify candidates into:

• Selected = **Yes** → Candidate suitable for hiring

• Selected = **No** → Candidate not suitable for hiring

This model can help organizations streamline the hiring process by automatically shortlisting candidates, reducing screening time, improving decision accuracy, and enabling recruiters to focus on high-potential applicants.

In addition, the project integrates **LIME**

Lime is a (Local Interpretable Model-agnostic Explanations) to provide transparency in model predictions. LIME helps explain why a candidate was selected or rejected by highlighting the key features that influenced the decision. This improves trust in the system and allows recruiters to understand and justify the model’s output effectively.


These libraries are imported to handle data processing and visualization tasks in the project. Numpy and pandas are used for data manipulation, while matplotlib and seaborn are used to visualize data patterns and insights.


In [10]:
# Core libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

The dataset is loaded using pandas from a CSV file into a DataFrame for further analysis. The head() function is used to display the first few rows of the dataset to understand its structure and contents.


In [11]:
df=pd.read_csv('AI_Resume_Screening.csv')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'AI_Resume_Screening.csv'

  tail()    function is used to display the last few rows of the dataset. This helps in verifying the data structure and checking how the dataset ends.


In [12]:
df.tail()

NameError: name 'df' is not defined

 describe() function provides a statistical summary of the dataset, including measures such as mean, standard deviation, minimum, and maximum values. It helps in understanding the distribution and overall characteristics of numerical features.


In [ ]:
df.describe()

 info() function provides a summary of the dataset, including the number of entries, column names, data types, and non-null values. It helps in understanding the structure of the data and identifying missing values.


In [13]:
df.info()

NameError: name 'df' is not defined

isna().sum() function is used to check for missing values in each column of the dataset. It returns the total count of null values, helping identify columns that may require data cleaning or imputation.


In [ ]:
df.isna().sum()

Fills missing values in the Certifications column with "No Certification" to handle null entries and maintain data consistency. After that, checks for remaining missing values in the dataset to ensure proper data preprocessing.


In [14]:
''' “The ‘Certifications’ column had missing values, which I treated as a meaningful absence by imputing them with
‘No Certification’ instead of dropping rows to preserve data and capture real-world scenarios.” '''

df['Certifications'] = df['Certifications'].fillna("No Certification")
df.isna().sum()

NameError: name 'df' is not defined

Converts all column names to lowercase and replaces spaces with underscores to maintain consistency and make column names easier to access in code.


In [ ]:
df.columns =df.columns.str.lower()
df.columns =df.columns.str.replace(' ','_')

Renames specific columns to simpler and consistent names for easier usage in the model. Displays the updated column names to verify the changes.


In [ ]:
df = df.rename(columns={
    'salary_expectation_($)': 'salary_expectation',
    'experience_(years)': 'experience_years',
    'ai_score_(0-100)': 'ai_score'
})
df.columns

Combines the skills and certifications columns into a single text feature to create a unified representation for NLP processing. This helps models like TF-IDF capture relationships between skills and certifications and improve prediction performance.


In [15]:
#To create a unified text representation so NLP models like TF-IDF can capture relationships between skills and certifications and improve prediction performance.
df["combined_text"] = df["skills"] + " " + df["certifications"]
df.columns

NameError: name 'df' is not defined

Displays the first few rows of the updated dataset to verify that the combined_text column has been created correctly and the data transformations are applied.


In [ ]:
df.head()

Fills any remaining missing values in the dataset with 0 to ensure there are no null entries before model training and processing.


In [ ]:
df = df.fillna(0)

Visualizes the distribution of recruiter decisions using a bar chart to compare the number of selected and rejected candidates. Helps in understanding class balance in the dataset.


In [16]:
import matplotlib.pyplot as plt

df["recruiter_decision"].value_counts().plot(kind="bar")
plt.title("Hiring vs Reject")
plt.show()

NameError: name 'df' is not defined

Plots a histogram of salary expectations to observe the distribution of salary values among candidates. Helps in understanding data spread and identifying patterns or outliers.


In [ ]:
df["salary_expectation"].hist()
plt.title("Salary Distribution")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

df["ai_score"].hist()
plt.title("AI Score Distribution")
plt.show()

Plots a histogram of AI scores to visualize their distribution across candidates. Helps in understanding how scores are spread and identifying any patterns or skewness in the data.


In [ ]:
df.boxplot(column="ai_score", by="recruiter_decision")
plt.title("AI Score vs Hiring Decision")
plt.show()

Creates pairwise plots for experience, projects, and recruiter decision to visualize relationships between features. Uses color differentiation based on recruiter decision to observe patterns and separation between selected and rejected candidates.


In [ ]:
sns.pairplot(
    df[[
        "experience_years",
        "projects_count",
        "recruiter_decision"
    ]],
    hue="recruiter_decision"
)

Visualizes the count of candidates across different education levels and compares them based on recruiter decisions. Helps identify how education influences selection and rejection patterns.


In [17]:
sns.countplot(x="education", hue="recruiter_decision", data=df)
plt.xticks(rotation=45)
plt.title("Education vs Recruiter Decision")
plt.show()

NameError: name 'df' is not defined

Creates a scatter plot to analyze the relationship between experience and AI score, with color indicating recruiter decisions. Helps observe how these factors influence hiring outcomes.


In [ ]:
sns.scatterplot(
    x="experience_years",
    y="ai_score",
    hue="recruiter_decision",
    data=df
)
plt.title("Experience vs AI Score")
plt.show()

Removes unnecessary columns such as resume_id, name, and ai_score from the dataset to avoid irrelevant features and improve model performance. Uses errors="ignore" to prevent errors if any column is not present.


In [ ]:
df = df.drop(columns=["resume_id", "name", "ai_score"], errors="ignore")

Combines skills and certifications into a single text feature while handling missing values by replacing them with empty strings. Ensures a clean and complete text input for further NLP processing.


In [ ]:
df["combined_text"] = df["skills"].fillna("") + " " + df["certifications"].fillna("")

Converts the recruiter_decision column into numerical format by mapping "Reject" to 0 and "Hire" to 1. Prepares the target variable for use in classification models.


In [ ]:
df["recruiter_decision"] = df["recruiter_decision"].map({
    "Reject": 0,
    "Hire": 1
})

Computes the correlation between key numerical features and the target variable, then visualizes it using a heatmap. Helps in understanding relationships between variables and identifying features that influence the recruiter decision.


In [ ]:
corr = df[[
    "experience_years",
    "projects_count",
    "salary_expectation",
    "recruiter_decision"
]].corr()

sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

Defines the input features and target variable for the model. Includes text, numerical, and categorical features in X, while y represents the recruiter decision used for classification.

In [ ]:
X = df[[
    "combined_text",
    "experience_years",
    "projects_count",
    "education",
    "job_role",
    "salary_expectation"
]]

y = df["recruiter_decision"]

Transforms the combined text data into numerical features using TF-IDF vectorization. Limits the number of features to the top 1000 important terms to efficiently represent textual information for the model.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=1000)

text_features = tfidf.fit_transform(X["combined_text"])

Selects numerical features such as experience, projects count, and salary expectation from the dataset to be used directly in the model without further transformation.


In [ ]:
num_features = X[[
    "experience_years",
    "projects_count",
    "salary_expectation"
]]

Encodes categorical features such as education and job role into numerical format using OneHotEncoder. Converts each category into binary vectors so they can be used effectively in the machine learning model.


In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder()

cat_features = encoder.fit_transform(X[["education", "job_role"]])

Combines text, numerical, and encoded categorical features into a single dataset using hstack. Creates a unified feature matrix that can be used for training the machine learning model.


In [ ]:
from scipy.sparse import hstack

X_final = hstack([text_features, num_features, cat_features])

Splits the dataset into training and testing sets, allocating 80% of the data for training and 20% for testing. Ensures reproducibility using a fixed random state for consistent results.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, test_size=0.2, random_state=42
)

Trains a Logistic Regression model on the training data to perform binary classification of recruiter decisions. Increases the maximum number of iterations to ensure proper convergence during training.


In [18]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1500)
model.fit(X_train, y_train)

NameError: name 'X_train' is not defined

Generates predictions for the test dataset using the trained model. Used to evaluate how well the model performs on unseen data.


In [ ]:
y_pred = model.predict(X_test)

Predicts recruiter decisions for the test dataset using the trained model. Helps evaluate model performance on unseen data.


In [ ]:
from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred))

Displays precision, recall, F1-score, and support for each class to evaluate the model’s classification performance.


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

Computes and visualizes the confusion matrix to show correct and incorrect predictions made by the model. Helps in understanding how well the model distinguishes between selected and rejected candidates.


In [ ]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred)

plt.imshow(cm)
plt.title("Confusion Matrix")
plt.colorbar()
plt.show()

Processes new candidate input by transforming text, numerical, and categorical features using the same preprocessing steps as training. Combines all features into a single format and uses the trained model to predict whether the candidate will be hired or rejected.


In [ ]:
# Step 1: text
new_text = tfidf.transform([" Learning "])

# Step 2: numeric
import numpy as np
new_num = np.array([[1, 2 , 1000]])

# Step 3: categorical
new_cat = encoder.transform([["B.Sc", "Data Scientist"]])

# Step 4: convert to dense
new_text_dense = new_text.toarray()
new_cat_dense = new_cat.toarray()

# Step 5: combine
new_final = np.hstack([new_text_dense, new_num, new_cat_dense])

# Step 6: predict
prediction = model.predict(new_final)

print("Prediction (1=Hire, 0=Reject):", prediction[0])

Trains a Decision Tree classifier on the training data to model decision rules for predicting recruiter outcomes. Uses a fixed random state to ensure consistent and reproducible results.


In [ ]:
from sklearn.tree import DecisionTreeClassifier

model_dt = DecisionTreeClassifier(random_state=42)

model_dt.fit(X_train, y_train)



In [ ]:
y_pred_dt = model_dt.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt))
print(confusion_matrix(y_test, y_pred_dt))

In [ ]:
# Step 1: text
new_text = tfidf.transform([" Python machine Learning"])

# Step 2: numeric
import numpy as np
new_num = np.array([[3, 5, 60000]])

# Step 3: categorical
new_cat = encoder.transform([["B.Sc", "Data Scientist"]])

# Step 4: convert to dense
new_text_dense = new_text.toarray()
new_cat_dense = new_cat.toarray()

# Step 5: combine
new_final = np.hstack([new_text_dense, new_num, new_cat_dense])

# Step 6: predict
prediction = model.predict(new_final)

print("Prediction (1=Hire, 0=Reject):", prediction[0])

Trains a Random Forest classifier using multiple decision trees to improve prediction accuracy and reduce overfitting. Uses a fixed random state for consistent and reproducible results.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(n_estimators=100, random_state=42)

model_rf.fit(X_train, y_train)

In [ ]:
y_pred_rf = model_rf.predict(X_test)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))

In [ ]:
# Step 1: text
new_text = tfidf.transform(["Python SQL Machine Learning"])

# Step 2: numeric
import numpy as np
new_num = np.array([[3, 5, 60000]])

# Step 3: categorical
new_cat = encoder.transform([["B.Sc", "Data Scientist"]])

# Step 4: convert to dense
new_text_dense = new_text.toarray()
new_cat_dense = new_cat.toarray()

# Step 5: combine
new_final = np.hstack([new_text_dense, new_num, new_cat_dense])

# Step 6: predict
prediction = model.predict(new_final)

print("Prediction (1=Hire, 0=Reject):", prediction[0])

Installs the LIME library, which is used to provide explanations for machine learning model predictions and improve model interpretability.


In [ ]:
pip install lime


Creates a list of feature names by combining TF-IDF text features, numerical features, and encoded categorical feature names. Ensures proper labeling of features for model interpretation and explanation.


In [ ]:
feature_names = (
    tfidf.get_feature_names_out().tolist() +
    ["experience_years", "projects_count", "salary_expectation"] +
    encoder.get_feature_names_out(["education","job_role"]).tolist()
)

Processes new candidate input by applying the same preprocessing steps used during training, including text transformation, numerical input handling, and categorical encoding. Uses the trained model to predict whether the candidate will be hired or rejected. Integrates LIME to generate explanations by identifying key features that positively or negatively influenced the prediction, and presents the results in a human-readable format.


In [ ]:
# ---- INPUT ----
new_text = tfidf.transform(["Python Machine Learning SQL"])
import numpy as np
new_num = np.array([[3, 5, 60000]])
new_cat = encoder.transform([["B.Sc", "Data Scientist"]])

new_final = np.hstack([new_text.toarray(), new_num, new_cat.toarray()])

# ---- PREDICTION ----
pred = int(model.predict(new_final)[0])
print("Prediction:", pred)

# ---- LIME ----
from lime.lime_tabular import LimeTabularExplainer

explainer = LimeTabularExplainer(
    X_train.copy(),
    feature_names=feature_names,
    mode="classification"
)

exp = explainer.explain_instance(
    new_final[0],
    model.predict_proba,
    labels=(pred,)
)

explanation = exp.as_list(label=pred)

# ---- OUTPUT ----
pos = [f for f,v in explanation if v > 0]
neg = [f for f,v in explanation if v < 0]

if pred == 1:
    print("\nPrediction: HIRE ✅")
    print("Candidate selected because:")
    for r in pos[:3]:
        print("-", r)

    print("\nAffected by:")
    for r in neg[:3]:
        print("-", r)

else:
    print("\nPrediction: REJECT ❌")
    print("Candidate rejected because:")
    for r in neg[:3]:
        print("-", r)

    print("\nBut helped by:")
    for r in pos[:3]:
        print("-", r)

Creates a Gradio-based user interface for the AI-powered resume screening system, allowing users to input candidate details such as skills, experience, projects, education, job role, salary, and certifications. Processes the input using the same preprocessing steps as the trained model and predicts whether the candidate will be hired or rejected. Integrates LIME to generate explainable insights by highlighting key features that influenced the decision. Displays both the prediction and human-readable explanation in an interactive web interface, making the system user-friendly and transparent.


In [ ]:
import gradio as gr
from scipy.sparse import hstack
import numpy as np
from lime.lime_tabular import LimeTabularExplainer

# Ensure all necessary objects are loaded or defined from the notebook state
# tfidf, encoder, model, feature_names, X_train, df are available from the kernel state.

def predict_applicant(skills_input, experience_years, projects_count, education, job_role, salary_expectation, certifications_input):
    # Combine skills and certifications for text processing
    combined_text_input = skills_input + " " + certifications_input

    # Step 1: Text features (TF-IDF)
    new_text_features = tfidf.transform([combined_text_input])
    new_text_dense = new_text_features.toarray()

    # Step 2: Numerical features
    new_num_features = np.array([[experience_years, projects_count, salary_expectation]])

    # Step 3: Categorical features (One-Hot Encoding)
    new_cat_features = encoder.transform([[education, job_role]])
    new_cat_dense = new_cat_features.toarray()

    # Step 4: Combine all features
    new_final_features = np.hstack([new_text_dense, new_num_features, new_cat_dense])

    # Step 5: Make prediction
    prediction_label = int(model.predict(new_final_features)[0])
    prediction_text = "HIRE ✅" if prediction_label == 1 else "REJECT ❌"

    # Step 6: Generate LIME explanation
    # Convert X_train to a dense array for LimeTabularExplainer
    explainer = LimeTabularExplainer(
        training_data=X_train.toarray(),
        feature_names=feature_names,
        class_names=["REJECT", "HIRE"], # Match with prediction_label 0 and 1
        mode="classification"
    )

    exp = explainer.explain_instance(
        new_final_features[0],
        model.predict_proba,
        num_features=len(feature_names), # Number of features to include in explanation
        labels=(prediction_label,)
    )

    explanation_list = exp.as_list(label=prediction_label)

    explanation_html = f"<h3>Prediction: {prediction_text}</h3>"
    if prediction_label == 1:
        explanation_html += "<h4>Candidate selected because:</h4><ul>"
        pos_features = [f for f, v in explanation_list if v > 0]
        for r in pos_features[:3]: # Limit to top 3 positive features
            explanation_html += f"<li>{r}</li>"
        explanation_html += "</ul><h4>Affected by:</h4><ul>"
        neg_features = [f for f, v in explanation_list if v < 0]
        for r in neg_features[:3]: # Limit to top 3 negative features
            explanation_html += f"<li>{r}</li>"
        explanation_html += "</ul>"
    else:
        explanation_html += "<h4>Candidate rejected because:</h4><ul>"
        neg_features = [f for f, v in explanation_list if v < 0]
        for r in neg_features[:3]: # Limit to top 3 negative features
            explanation_html += f"<li>{r}</li>"
        explanation_html += "</ul><h4>But helped by:</h4><ul>"
        pos_features = [f for f, v in explanation_list if v > 0]
        for r in pos_features[:3]: # Limit to top 3 positive features
            explanation_html += f"<li>{r}</li>"
        explanation_html += "</ul>"

    return prediction_text, explanation_html

# Get unique values for dropdowns from the 'df' DataFrame
education_choices = df['education'].unique().tolist()
job_role_choices = df['job_role'].unique().tolist()

# Define Gradio interface
iface = gr.Interface(
    fn=predict_applicant,
    inputs=[
        gr.Textbox(label="Skills (comma-separated)", placeholder="e.g., Python, Machine Learning, SQL"),
        gr.Number(label="Experience (Years)", value=3),
        gr.Number(label="Projects Count", value=5),
        gr.Dropdown(label="Education", choices=education_choices, value=education_choices[0]),
        gr.Dropdown(label="Job Role", choices=job_role_choices, value=job_role_choices[0]),
        gr.Number(label="Salary Expectation ($)", value=60000),
        gr.Textbox(label="Certifications (comma-separated)", placeholder="e.g., Google ML, AWS Certified")
    ],
    outputs=[
        gr.Textbox(label="Recruiter Decision"),
        gr.HTML(label="Explanation (LIME)")
    ],
    title="AI Powered Resume Screening & Candidate Selection",
    description="Predicts whether a candidate should be hired or rejected and provides an explanation using LIME.",
    theme=gr.themes.Soft()
)

iface.launch()

**Conclusion**

In this project, we developed a machine learning model to predict whether a candidate should be selected or rejected using an AI-powered resume screening system.

Key steps performed in this project:

• Data cleaning and preprocessing
• Handling missing values
• Exploratory Data Analysis
• Text processing using TF-IDF
• Encoding categorical features
• Combining text, numerical, and categorical data
• Model training using multiple algorithms
• Model evaluation using accuracy, classification report, and confusion matrix
• Integration of LIME for model explanation

Among the tested models, Random Forest provided the best performance in predicting recruiter decisions.

The model can help organizations:

• Automate resume screening process
• Identify suitable candidates efficiently
• Reduce manual effort and hiring time
• Improve decision-making using data-driven insights
• Provide transparent explanations using LIME

This project demonstrates how machine learning and explainable AI can be applied to build an intelligent and reliable recruitment system for real-world applications.
